# 06 — Building annotations from a video with a large teacher

This notebook turns a video into a candidate for the future `v003` dataset. The large model is not meant for the iPhone: it only proposes annotations that we then correct by hand. The nano model already adapted to `v002` complements its proposals.

Pipeline:

1. keep one source frame out of four;
2. have `yolo26x` and our fine-tuned nano propose boxes;
3. convert the COCO class `sports ball` into the PitchLens class `ball`;
4. merge the two results, removing overlapping boxes;
5. record image, timestamp, confidence, originating teacher and box;
6. review each proposal in the same interface as for `v002`.

The pilot selects 100 frames spread over the **whole** video. It does not simply take the first 100.

In [ ]:
%matplotlib widget

import importlib
import json
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import torch
from IPython.display import Image as DisplayImage
from IPython.display import Markdown, display
from PIL import Image, ImageDraw

PROJECT_ROOT = next(
    folder
    for folder in (Path.cwd().resolve(), Path.cwd().resolve().parent)
    if (folder / "pyproject.toml").exists()
)
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pitchlens.data.pseudo_label_video as pseudo_label_module

# Important in a notebook: reload the module after a code fix.
pseudo_label_module = importlib.reload(pseudo_label_module)
combine_teacher_datasets = pseudo_label_module.combine_teacher_datasets
expected_sample_count = pseudo_label_module.expected_sample_count
prepare_video_pseudo_labels = pseudo_label_module.prepare_video_pseudo_labels

print(f"Project: {PROJECT_ROOT}")

## 1. Experiment parameters

`PILOT = True` only produces 100 frames, spread from start to end. After inspection, switching to `False` will produce the 1,232 frames spaced four frames apart in another folder.

The `0.10` threshold is deliberately low: during annotation, we prefer to propose a doubtful ball than to miss a ball in the air. The false proposals can be removed.

In [ ]:
VIDEO_PATH = PROJECT_ROOT / "data/external/videos/inference_input_yt.mp4"
LARGE_TEACHER = (
    PROJECT_ROOT
    / "models/checkpoints/teachers/yolo26x_v002_hf/highres_960/best.pt"
)
DOMAIN_TEACHER = (
    PROJECT_ROOT
    / "models/checkpoints/baseline_runs/yolo26n_v002_local_20260918_002618"
    / "train/weights/best.pt"
)

PILOT = True
FRAME_STRIDE = 4
MAX_SAMPLES = 100 if PILOT else None
# The teacher was continued at 960 px: this is its reference resolution.
IMAGE_SIZE = 960
CONFIDENCE = 0.10
MAX_DETECTIONS = 5
SOURCE_CLASS_ID = 0  # the fine-tuned teacher only contains the ball class
# YOLO26x at 960 px crashes the kernel on this 8 GB Mac.
# Its proposals are therefore computed on Hugging Face and then downloaded here.
LARGE_INFERENCE_BACKEND = "Hugging Face A10G"

suffix = "pilot100" if PILOT else "full"
LARGE_OUTPUT_ROOT = (
    PROJECT_ROOT
    / "data/interim"
    / f"video_teacher_{VIDEO_PATH.stem}_{suffix}_yolo26x_highres960"
)
DOMAIN_OUTPUT_ROOT = (
    PROJECT_ROOT
    / "data/interim"
    / f"video_teacher_{VIDEO_PATH.stem}_{suffix}_yolo26n_finetuned"
)
ENSEMBLE_OUTPUT_ROOT = (
    PROJECT_ROOT
    / "data/interim"
    / f"video_teacher_{VIDEO_PATH.stem}_{suffix}_ensemble_highres960"
)

capture = cv2.VideoCapture(str(VIDEO_PATH))
if not capture.isOpened():
    raise FileNotFoundError(f"Unreadable video: {VIDEO_PATH}")
total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
fps = float(capture.get(cv2.CAP_PROP_FPS) or 30.0)
capture.release()
full_count = expected_sample_count(total_frames, FRAME_STRIDE)
selected_count = min(full_count, MAX_SAMPLES) if MAX_SAMPLES else full_count

display(Markdown(
    f"**Video:** `{VIDEO_PATH.name}` — {total_frames:,} frames — {total_frames / fps:.1f} s  \\n"
    f"**Selection:** {selected_count:,} frames out of {full_count:,} possible  \\n"
    f"**Teachers:** adapted `{LARGE_TEACHER.name}` (primary) + adapted nano (complement) at {IMAGE_SIZE}px  \\n"
    f"**Large teacher:** `{LARGE_INFERENCE_BACKEND}`  \\n"
    f"**Output to review:** `{ENSEMBLE_OUTPUT_ROOT}`"
))

## 2. Extract and pseudo-label

The large `yolo26x` fine-tuned on PitchLens takes priority when a box from both models overlaps; the adapted nano only adds its complementary proposals. The heavy computation of the large model is done on a Hugging Face GPU, then this notebook reads back the downloaded results. It therefore does not load the 960 px model into the Mac's 8 GB. The results are immutable: a re-run replaces neither the images nor your decisions.

In [ ]:
large_report_path = LARGE_OUTPUT_ROOT / "pseudo_label_report.json"
if large_report_path.exists():
    large_report = json.loads(large_report_path.read_text())
    print("yolo26x proposals already present: no file replaced.")
elif LARGE_INFERENCE_BACKEND == "Hugging Face A10G":
    raise RuntimeError(
        "The large teacher's proposals must first be computed on "
        "Hugging Face: loading it locally exceeds the Mac's 8 GB."
    )
else:
    large_report = prepare_video_pseudo_labels(
        project_root=PROJECT_ROOT,
        video_path=VIDEO_PATH,
        output_root=LARGE_OUTPUT_ROOT,
        teacher_model=LARGE_TEACHER,
        frame_stride=FRAME_STRIDE,
        max_samples=MAX_SAMPLES,
        image_size=IMAGE_SIZE,
        confidence=CONFIDENCE,
        max_detections=MAX_DETECTIONS,
        source_class_id=SOURCE_CLASS_ID,
        device="cpu",
    )

domain_report_path = DOMAIN_OUTPUT_ROOT / "pseudo_label_report.json"
if domain_report_path.exists():
    domain_report = json.loads(domain_report_path.read_text())
    print("Adapted nano proposals already present: no file replaced.")
else:
    domain_report = prepare_video_pseudo_labels(
        project_root=PROJECT_ROOT,
        video_path=VIDEO_PATH,
        output_root=DOMAIN_OUTPUT_ROOT,
        teacher_model=DOMAIN_TEACHER,
        frame_stride=FRAME_STRIDE,
        max_samples=MAX_SAMPLES,
        image_size=640,
        confidence=CONFIDENCE,
        max_detections=MAX_DETECTIONS,
        source_class_id=0,
        device="mps" if torch.backends.mps.is_available() else "cpu",
    )

ensemble_report_path = ENSEMBLE_OUTPUT_ROOT / "pseudo_label_report.json"
if ensemble_report_path.exists():
    report = json.loads(ensemble_report_path.read_text())
    print("Merge already present: no box or decision replaced.")
else:
    report = combine_teacher_datasets(
        project_root=PROJECT_ROOT,
        # The adapted large teacher takes priority; the nano complements it.
        source_roots=[LARGE_OUTPUT_ROOT, DOMAIN_OUTPUT_ROOT],
        output_root=ENSEMBLE_OUTPUT_ROOT,
        iou_threshold=0.5,
    )
OUTPUT_ROOT = ENSEMBLE_OUTPUT_ROOT

display(Markdown(
    f"### Merged teachers result\n"
    f"- Frames extracted: **{report['sampled_images']:,}**\n"
    f"- Frames with at least one box: **{report['frames_with_teacher_box']:,}**\n"
    f"- Frames with no proposed box: **{report['frames_without_teacher_box']:,}**\n"
    f"- Boxes proposed after deduplication: **{report['teacher_boxes']:,}**\n"
    f"- Contributions kept: **{report['kept_boxes_by_teacher']}**"
))

## 3. Preview of the proposals

Red = merged proposal from the teachers. This preview is only used to decide whether the system is useful enough before producing the full version.

In [ ]:
import csv

with (OUTPUT_ROOT / "manifest.csv").open(newline="") as stream:
    manifest = list(csv.DictReader(stream))

preview_indices = [round(i * (len(manifest) - 1) / 11) for i in range(12)]
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for axis, index in zip(axes.flat, preview_indices):
    row = manifest[index]
    image = Image.open(PROJECT_ROOT / row["source_image"]).convert("RGB")
    draw = ImageDraw.Draw(image)
    label_path = PROJECT_ROOT / row["source_label"]
    for line in label_path.read_text().splitlines():
        _, x, y, width, height = map(float, line.split())
        left = (x - width / 2) * image.width
        top = (y - height / 2) * image.height
        right = (x + width / 2) * image.width
        bottom = (y + height / 2) * image.height
        draw.rectangle((left, top, right, bottom), outline="red", width=4)
    axis.imshow(image)
    axis.set_title(
        f"t={float(row['timestamp_seconds']):.1f}s · {row['teacher_boxes']} box(es)",
        fontsize=9,
    )
    axis.axis("off")
fig.suptitle("Merged proposals spread across the video")
fig.tight_layout()
plt.show()

## 4. Correct the proposals

The red rectangles are the merged proposals from the adapted large `yolo26x` and the adapted nano, not yet ground truth.

- **Confirm and advance**: all displayed boxes are correct;
- **No ball**: remove all the proposals and keep a negative image;
- **Keep only this box**: keep a single proposal among several;
- **Save the drawn box**: replace all the proposals with your box;
- **Exclude the image**: unusable or ambiguous image.

The decisions are recorded in this intermediate dataset. Re-running the notebook does not erase them.

In [ ]:
import pitchlens.data.review as review_module

review_module = importlib.reload(review_module)
store = review_module.ReviewStore(OUTPUT_ROOT)
store.progress()

In [ ]:
# Close the previous interface before recreating it when this cell is re-run.
previous_review = globals().get("review")
if previous_review is not None:
    previous_review.selector.disconnect_events()
    previous_review.plt.close(previous_review.figure)
    previous_review.ui.close()
review = review_module.ReviewWidget(store)
review.display()

## 5. Human review progress

The full dataset should only be generated after the pilot has been judged useful enough. The images of this video will stay in the train split; another complete video will have to serve as domain validation/test.

In [ ]:
store.progress()

## 6. Controlled benchmark before the full version

The choice of a teacher must be based on metrics, not on its size. We therefore compare the adapted nano, the raw COCO `yolo26x` and the adapted `yolo26x` on the **670 v002 validation images**, all at 640 px and with the same evaluator.

The test split stays locked: it will only be used after the final choice of the pipeline. Using the test split to choose the teacher would turn it into a second validation set and make its estimate too optimistic.

In [ ]:
benchmark_path = PROJECT_ROOT / "models/reports/teacher_benchmark_v002_val/comparison.json"
if not benchmark_path.exists():
    display(Markdown(
        "Benchmark still running or not started. Reproducible command:  \\n"
        "`python -m pitchlens.training.benchmark_teachers`"
    ))
else:
    benchmark = json.loads(benchmark_path.read_text())
    lines = [
        "| Model | mAP50 | mAP50–95 | Precision | Recall | F1 | FP on negatives | Device |",
        "|---|---:|---:|---:|---:|---:|---:|---|",
    ]
    for model_name, result in benchmark["models"].items():
        fixed = result["fixed"]
        lines.append(
            f"| {model_name} | {result['map50']:.1%} | {result['map50_95']:.1%} | "
            f"{fixed['precision']:.1%} | {fixed['recall']:.1%} | {fixed['f1']:.1%} | "
            f"{fixed['fp_on_negatives']} | {result['device']} |"
        )
    display(Markdown("\n".join(lines)))
    benchmark_figure = benchmark_path.parent / "comparison.png"
    if benchmark_figure.exists():
        display(DisplayImage(filename=str(benchmark_figure)))

## 7. Same teacher, different inference resolution

Before retraining at high resolution, we keep exactly the same `best.pt` trained at 640 px and only evaluate its behavior at 640, 960 and 1280 px. The split, the 670 images, the batch and the thresholds stay identical.

The recall by size groups the ground truths by the length of their longest side scaled to a 640 px image. This directly shows the effect on very small balls.

In [ ]:
resolution_path = (
    PROJECT_ROOT / "models/reports/teacher_resolution_benchmark/comparison.json"
)
if not resolution_path.exists():
    display(Markdown("The resolution benchmark is not available yet."))
else:
    resolution_report = json.loads(resolution_path.read_text())
    resolution_results = resolution_report["results"]
    lines = [
        "| Resolution | mAP50 | mAP50–95 | Precision | Recall | F1 | Tiny recall <8 px | FP on negatives | Inference/image |",
        "|---:|---:|---:|---:|---:|---:|---:|---:|---:|",
    ]
    for resolution, result in resolution_results.items():
        fixed = result["fixed"]
        tiny = result["recall_by_size"]["tiny_<8px"]["recall"]
        inference = result["speed_ms_per_image"]["inference"]
        lines.append(
            f"| {resolution} | {result['map50']:.1%} | {result['map50_95']:.1%} | "
            f"{fixed['precision']:.1%} | {fixed['recall']:.1%} | {fixed['f1']:.1%} | "
            f"{tiny:.1%} | {fixed['fp_on_negatives']} | {inference:.1f} ms |"
        )
    display(Markdown("\n".join(lines)))
    display(DisplayImage(filename=str(resolution_path.parent / "comparison.png")))

    baseline = resolution_results["640"]
    candidate = resolution_results["960"]
    tiny_640 = baseline["recall_by_size"]["tiny_<8px"]
    tiny_960 = candidate["recall_by_size"]["tiny_<8px"]
    display(Markdown(
        "**Reading:** at 960 px, the teacher finds "
        f"**{tiny_960['matched'] - tiny_640['matched']} additional tiny balls** "
        f"and gains {(candidate['fixed']['recall'] - baseline['fixed']['recall']):+.1%} of overall recall. "
        "However, its false positives increase and its mAP50–95 drops: "
        "960 px is therefore a good candidate resolution for a new training run, "
        "but not yet an automatic replacement for the 640 px teacher."
    ))

## 8. Teacher retrained at 960 px

The 640 teacher was continued for 12 epochs at 960 px on the A10G GPU. `best.pt` corresponds to the best validation state, around epoch 9: the last epochs are kept in the curves but do not automatically replace the best weights.

The comparison below clearly distinguishes three situations: the 640 model used at 640, the 640 model simply upscaled to 960, and finally the model actually retrained and used at 960.

In [ ]:
highres_report_path = (
    PROJECT_ROOT / "models/reports/teacher_highres_960/training_report.json"
)
if not highres_report_path.exists():
    display(Markdown("The high-resolution training is not available."))
else:
    highres_report = json.loads(highres_report_path.read_text())
    old_results = json.loads(resolution_path.read_text())["results"]
    new_results = highres_report["validation_after"]
    compared = [
        ("Teacher 640 @ 640", old_results["640"]),
        ("Teacher 640 @ 960", old_results["960"]),
        ("Teacher 960 @ 960", new_results["960"]),
    ]
    lines = [
        "| Experiment | mAP50 | mAP50–95 | Precision | Recall | F1 | TP | FP | FN | FP on negatives |",
        "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|",
    ]
    for label, result in compared:
        fixed = result["fixed"]
        lines.append(
            f"| {label} | {result['map50']:.1%} | {result['map50_95']:.1%} | "
            f"{fixed['precision']:.1%} | {fixed['recall']:.1%} | {fixed['f1']:.1%} | "
            f"{fixed['tp']} | {fixed['fp']} | {fixed['fn']} | {fixed['fp_on_negatives']} |"
        )
    display(Markdown("\n".join(lines)))
    highres_root = highres_report_path.parent
    display(DisplayImage(filename=str(highres_root / "comparison_old_vs_new.png")))
    display(DisplayImage(filename=str(highres_root / "train/results.png")))

    previous = old_results["960"]["fixed"]
    current = new_results["960"]["fixed"]
    display(Markdown(
        "**Reading:** at the same 960 px resolution, retraining adds "
        f"**{current['tp'] - previous['tp']} correct detections**, removes "
        f"**{previous['fp'] - current['fp']} false detections** and raises the F1 from "
        f"**{previous['f1']:.1%} to {current['f1']:.1%}**. "
        "This new weight therefore becomes the primary teacher for the next video pilot."
    ))